# 실행 경로 비교 — 동기 vs 배치

같은 100건을 **같은 앵커 풀·같은 프롬프트·같은 모델**로 두 번 라벨링했습니다.
다른 것은 실행 경로 하나뿐입니다.

| | 실행 | 구조화 출력 |
|---|---|---|
| 동기 | `run_claude_labeling.py` | `messages.parse(output_format=LabelResult)` |
| 배치 | `run_claude_batch.py` | 요청 본문에 `output_config.format` 직접 지정 |

`messages.parse()`는 `output_config.format` 위에 얹힌 SDK 헬퍼이므로 원리상 같은 메커니즘이지만,
실제로 같은 결과가 나오는지는 확인해야 합니다.

## 왜 확인하는가

전수 1,024건 중 Chunk 1(100건)만 동기로 돌았고 나머지 924건은 배치로 돌았습니다.
그 100건은 신용회복위원회 문서에 몰려 있어(95건), 만약 실행 경로가 라벨에 영향을 준다면
**한 문서에 계통 오차**로 나타납니다. 문서 단위로 학습·평가를 나누는 설계(§10.1)에서는
무시하기 어려운 문제입니다.

## 해석 기준 — 얼마나 달라야 "다른 것"인가

같은 조건으로 3회 반복해도 라벨은 원래 흔들립니다. 결정 23에서 측정한 값이 기준선입니다.

| 조건 | 주 라벨 3/3 일치율 |
|---|---:|
| zero-shot | 90.0% |
| few-shot 층화 | 92.5% |

즉 **같은 방식으로 두 번 돌려도 7~10%는 달라집니다.**

- 일치율이 92.5% 이상이면 → 실행 경로 차이가 아니라 모델의 통상적인 실행 간 변동
- 92.5% 미만이면 → 반복 변동보다 낮으므로 방식 차이를 의심할 근거

이 기준선이 없으면 "몇 건 달라졌다"를 놓고 판단할 방법이 없습니다.

In [ ]:
from pathlib import Path
from collections import Counter
import json

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next(
    f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
    if f in installed
)
plt.rcParams['axes.unicode_minus'] = False

LABELS = ['통상수용', '견적반영', '계약·질의검토']
AUX = ['cost_basis', 'domain_dependency', 'build_difficulty']
# 결정 23에서 측정한 같은 방식 3회 반복 일치율. 판단 기준선입니다.
REPEAT_BASELINE = 0.925

RUNS = {
    '동기': ROOT / 'reports/current/claude_runs/full_requirements_v0.2.0_fewshot_v5/results.jsonl',
    '배치': ROOT / 'reports/current/claude_runs/batch_chunk1_redo/results.jsonl',
}


def load(path):
    if not path.exists():
        raise FileNotFoundError(
            f'{path.name}이 없습니다. 배치가 아직 처리 중이면 먼저 받으세요:\n'
            '  python -m scripts.labeling.run_claude_batch --download '
            '--batch-dir reports/current/claude_runs/batch_chunk1_redo'
        )
    rows = [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]
    return {r['requirement_uid']: r for r in rows if r.get('status') == 'ok'}

runs = {name: load(path) for name, path in RUNS.items()}
shared = sorted(set(runs['동기']) & set(runs['배치']))

for name, data in runs.items():
    print(f'{name}: {len(data)}건 성공')
print(f'공통: {len(shared)}건')

# 비교표. 한 행이 요구사항 하나, 열이 두 실행의 판정입니다.
df = pd.DataFrame([
    {
        'uid': uid,
        '동기': runs['동기'][uid]['label']['primary_action'],
        '배치': runs['배치'][uid]['label']['primary_action'],
        **{f'{f}_동기': runs['동기'][uid]['label'][f] for f in AUX},
        **{f'{f}_배치': runs['배치'][uid]['label'][f] for f in AUX},
        'blockers_동기': tuple(sorted(runs['동기'][uid]['label']['blockers'])),
        'blockers_배치': tuple(sorted(runs['배치'][uid]['label']['blockers'])),
    }
    for uid in shared
])
print(f'비교 대상 {len(df)}건')

In [ ]:
# 주 라벨 일치율과 전이 행렬.
# 전이 행렬의 대각선이 일치, 비대각이 판정이 바뀐 건입니다.
match = (df['동기'] == df['배치'])
rate = match.mean()
print(f'주 라벨 일치: {match.sum()}/{len(df)} = {rate:.1%}')
print(f'판정 변동: {(~match).sum()}건\n')

matrix = pd.crosstab(df['동기'], df['배치']).reindex(index=LABELS, columns=LABELS, fill_value=0)
matrix.index.name = '동기 \\ 배치'
display(matrix)

# 분포가 한쪽으로 쏠렸는지 확인합니다.
# 전이가 양방향으로 상쇄되면 분포는 같아도 개별 판정은 달라집니다.
dist = pd.DataFrame({
    '동기': df['동기'].value_counts().reindex(LABELS, fill_value=0),
    '배치': df['배치'].value_counts().reindex(LABELS, fill_value=0),
})
dist['차이'] = dist['배치'] - dist['동기']
display(dist)

In [ ]:
# 보조 축 일치율. 주 라벨보다 흔들리기 쉬운 값들입니다.
# 결정 22에서 blockers 조합의 3회 반복 일치율이 40%대로 관측된 바 있습니다.
rows = [{'축': '주 라벨(primary_action)', '일치': int(match.sum()), '전체': len(df), '일치율': rate}]
for field in AUX:
    same = (df[f'{field}_동기'] == df[f'{field}_배치'])
    rows.append({'축': field, '일치': int(same.sum()), '전체': len(df), '일치율': same.mean()})
same_blockers = (df['blockers_동기'] == df['blockers_배치'])
rows.append({'축': 'blockers (조합 전체)', '일치': int(same_blockers.sum()), '전체': len(df), '일치율': same_blockers.mean()})

summary = pd.DataFrame(rows)
summary['일치율'] = (summary['일치율'] * 100).round(1)
display(summary.set_index('축'))

print('보조 축이 주 라벨보다 낮게 나오는 것은 정상입니다.')
print('주 라벨은 3분류지만 blockers는 32가지 조합이 가능하므로 우연 일치 확률부터 다릅니다.')

In [ ]:
# 판정이 바뀐 건의 상세. 어떤 방향으로 바뀌었고 근거가 어떻게 달라졌는지 봅니다.
changed = df[~match]
if changed.empty:
    print('판정이 바뀐 건 없음')
else:
    for _, row in changed.iterrows():
        s, b = runs['동기'][row['uid']]['label'], runs['배치'][row['uid']]['label']
        print(f"\n{row['uid']}")
        print(f"  동기: {s['primary_action']:<12} blockers={s['blockers']} cost={s['cost_basis']}")
        print(f"  배치: {b['primary_action']:<12} blockers={b['blockers']} cost={b['cost_basis']}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

dist[['동기', '배치']].plot.bar(ax=axes[0], rot=0, color=['#4C78A8', '#F58518'])
axes[0].set(title='주 라벨 분포 비교', ylabel='건수', xlabel='')

axes[1].barh(summary['축'], summary['일치율'], color='#54A24B')
axes[1].axvline(REPEAT_BASELINE * 100, color='crimson', linestyle='--',
                label=f'반복 실행 기준선 {REPEAT_BASELINE:.1%}')
axes[1].set(title='축별 일치율', xlabel='일치율 (%)', xlim=(0, 100))
axes[1].legend()
plt.tight_layout()

# 최종 판단. 기준선은 결정 23에서 측정한 같은 방식 3회 반복 일치율입니다.
print('\n' + '=' * 60)
if rate >= REPEAT_BASELINE:
    print(f'주 라벨 일치 {rate:.1%} >= 기준선 {REPEAT_BASELINE:.1%}')
    print('실행 경로 차이가 아니라 모델의 통상적인 실행 간 변동으로 봅니다.')
    print('어느 쪽 결과를 채택해도 무방하며, 배치로 통일하면 전수가 단일 경로가 됩니다.')
else:
    print(f'주 라벨 일치 {rate:.1%} < 기준선 {REPEAT_BASELINE:.1%}')
    print('반복 변동 범위보다 낮습니다. 실행 경로가 라벨에 영향을 준다고 의심할 근거가 됩니다.')
    print('전이 행렬이 한 방향으로 쏠렸는지 확인하고, 쏠렸다면 그 방향이 왜 생기는지 봐야 합니다.')